# Evaluate DDPG on Fixed Obstacles

Load the saved DDPG checkpoint and evaluate it on a fixed unseen-obstacle configuration.

## Imports and Paths

In [ ]:
from pathlib import Path
import sys

import torch

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()
    if BASE_DIR.name != "Continuous_Diff_Drive":
        BASE_DIR = BASE_DIR / "Continuous_Diff_Drive"

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from diff_drive_agent import DiffDriveAgent
from diff_drive_env import DiffDriveEnv

BASE_DIR

## Fixed Evaluation Configuration

In [ ]:
CHECKPOINT_PATH = BASE_DIR / "models" / "ddpg_checkpoint.pt"
EVALUATION_VIDEO_DIR = BASE_DIR / "videos" / "evaluation"
EVALUATION_NAME_PREFIX = "ddpg_diff_drive_eval_unseen_fixed_obstacles_greedy"

# A room with a few axis-aligned rectangular obstacles.
# Each obstacle is (x, y, width, height) in metres, origin at bottom-left.
OBSTACLES = [
    (2.0, 4.0, 3.0, 0.3),
    (5.0, 2.0, 0.3, 3.0),
    (7.0, 6.0, 1.5, 0.3),
]

ENV_KWARGS = dict(
    room_size       = (10.0, 10.0),
    obstacles       = OBSTACLES,
    random_obst     = False,
    robot_start     = (1.0, 1.0),
    goal_pos        = (8.5, 8.5),
    max_step        = 1000,
    n_lidar_rays    = 16,
    lidar_max_range = 5.0,
    robot_radius    = 0.3,
    dt              = 0.1,
    render_mode     = "rgb_array",
)

ACTOR_LR      = 1e-4
CRITIC_LR     = 1e-3
DISCOUNT      = 0.95
TAU           = 0.005
NOISE_STD     = 0.2
NOISE_CLIP    = 0.5
BATCH_SIZE    = 256
BUFFER_SIZE   = 100_000
HIDDEN_DIM    = 128
WARMUP_STEPS  = 1_000
DEVICE        = "cpu"
N_EPISODES    = 3

EVALUATION_VIDEO_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH

## Environment and Agent

In [ ]:
env = DiffDriveEnv(**ENV_KWARGS)

agent = DiffDriveAgent(
    env          = env,
    actor_lr     = ACTOR_LR,
    critic_lr    = CRITIC_LR,
    discount     = DISCOUNT,
    tau          = TAU,
    noise_std    = NOISE_STD,
    noise_clip   = NOISE_CLIP,
    batch_size   = BATCH_SIZE,
    buffer_size  = BUFFER_SIZE,
    hidden_dim   = HIDDEN_DIM,
    warmup_steps = WARMUP_STEPS,
    device       = DEVICE,
)

agent

## Load Checkpoint

In [ ]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"No checkpoint found at {CHECKPOINT_PATH}")

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
agent.actor.load_state_dict(checkpoint["actor"])
agent.critic.load_state_dict(checkpoint["critic"])
agent.actor_target.load_state_dict(checkpoint["actor_target"])
agent.critic_target.load_state_dict(checkpoint["critic_target"])

print(f"Loaded checkpoint from {CHECKPOINT_PATH}")

## Greedy Evaluation

In [ ]:
agent.eval_recorded(
    video_folder = EVALUATION_VIDEO_DIR,
    name_prefix  = EVALUATION_NAME_PREFIX,
    n_episodes   = N_EPISODES,
    add_noise    = False,
)

## Evaluation Videos

In [ ]:
from IPython.display import Video, display

video_paths = sorted(EVALUATION_VIDEO_DIR.glob(f"{EVALUATION_NAME_PREFIX}*.mp4"))

if not video_paths:
    print(f"No videos found yet in {EVALUATION_VIDEO_DIR}")

for video_path in video_paths:
    print(video_path.name)
    display(Video(filename=str(video_path), embed=True))